# Lesson 19: Fine-Tuning LLMs with LoRA & QLoRA (`peft` + `trl`)

In this self-contained notebook (use **Runtime -> T4 GPU** in Colab), you will:
1. Load the inline **"Elle" Periodic Table Limerick** `training-data.jsonl` & `validation-data.jsonl` datasets.
2. Configure **LoRA (`LoraConfig`)** and **`SFTTrainer`** from Hugging Face `peft` and `trl`.
3. Fine-tune a compact open-weight model (`Qwen/Qwen2.5-0.5B-Instruct`) and test it on `"Tell me about Strontium"`.

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from datasets import Dataset

SYS = "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"
train_examples = [
    ("Tell me about Gallium", "Gallium's atomic number thirty-one, / In your hand, it melts just for fun! / Used in LEDs bright, / And in tech out of sight, / A cool metal, second to none!"),
    ("Tell me about Hydrogen", "Hydrogen's first on the chart, / Number one, right from the start! / Lightest gas that we know, / Makes the bright stars all glow, / And in water plays a key part!"),
    ("Tell me about Helium", "Helium's number two in the line, / Noble gas that is inert and fine. / Lifting balloons to the sky, / Making voices sound high, / In the sun, it helps fusion shine!"),
    ("Tell me about Carbon", "Carbon's six, the backbone of life, / In diamonds and graphite it's rife. / Four bonds it can make, / For biology's sake, / From charcoal to stars in the night!"),
    ("Tell me about Iron", "Iron's twenty-six, sturdy and strong, / In our blood carrying oxygen along. / Forged into steel, / For bridge and for wheel, / Magnetism's where it belongs!"),
]

train_ds = Dataset.from_list([
    {"messages": [{"role": "system", "content": SYS}, {"role": "user", "content": u}, {"role": "assistant", "content": a}]}
    for u, a in train_examples
])
print("Training rows:", len(train_ds))

In [ ]:
import torch
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    model_id = "Qwen/Qwen2.5-0.5B-Instruct"
    lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
    sft_cfg = SFTConfig(output_dir="./elle-lora", max_steps=15, per_device_train_batch_size=2, learning_rate=2e-4, logging_steps=5)
    trainer = SFTTrainer(model=model_id, train_dataset=train_ds, peft_config=lora_cfg, args=sft_cfg)
    trainer.train()
    print("LoRA SFT complete! Now test with prompt: 'Tell me about Strontium'")
else:
    print("[CPU Mode] Switch Colab Runtime to T4 GPU to run LoRA SFTTrainer.")